In [1]:
import pandas as pd
import numpy as np
import re

PATH_MASTER = r"../../data/Riot_API/feature/datamart/dm_match_team_all.csv"
df = pd.read_csv(PATH_MASTER)

print("master shape:", df.shape)
print(df[["match_id","team_id","win_team","queue_id","game_version"]].head(3))
cols = df.columns.tolist()

def has(c): 
    return c in df.columns

master shape: (108412, 1254)
         match_id  team_id  win_team  queue_id    game_version
0  BR1_3165661977      100      True       420  15.22.724.5161
1  BR1_3165661977      200     False       420  15.22.724.5161
2  BR1_3200312712      100     False       420   16.2.741.3171


In [2]:
rows_per_match = df.groupby("match_id").size().value_counts().head(10)
print(rows_per_match)

# 2row가 아닌 match가 있다면 feature pool 전에 정리 필요
bad_matches = df.groupby("match_id").size()
bad_matches = bad_matches[bad_matches != 2]
print("non-2row matches:", len(bad_matches))
bad_matches.head()

2    54206
Name: count, dtype: int64
non-2row matches: 0


Series([], dtype: int64)

In [3]:
def add_diff_from_total(df, col, new_col):
    """
    match_id당 2row라는 전제.
    diff = our_value - enemy_value
    """
    if col not in df.columns:
        return None
    s = df.groupby("match_id")[col].transform("sum")
    df[new_col] = df[col] - (s - df[col])
    return new_col

In [4]:
context_cols = [c for c in [
    "match_id",
    "platform",
    "team_id",
    "win_team",
    "queue_id",
    "game_version",
    "game_creation",
    "game_duration",
] if has(c)]

print("context_cols:", context_cols)

context_cols: ['match_id', 'platform', 'team_id', 'win_team', 'queue_id', 'game_version', 'game_creation', 'game_duration']


In [ ]:
team_diff_cols = []

# (A) Economy / Combat / Vision
for c in [
    "gold_earned_diff",
    "gold_spent_diff",
    "kills_diff",
    "deaths_diff",
    "assists_diff",
    "vision_score_diff",
    "wards_placed_diff",
    "wards_killed_diff",
]:
    if has(c):
        team_diff_cols.append(c)

# (B) Towers/Structure
for c in [
    "turret_takedowns_diff",
    "turret_plates_taken_diff",
    "inhibitor_takedowns_diff",
]:
    if has(c):
        team_diff_cols.append(c)

# (C) Epic objectives (diff가 이미 있으면 사용)
for c in [
    "dragon_kills_diff",
    "baron_kills_diff",
    "atakhan_kills_diff",         # 2025만 존재, 2026은 0으로 채워두는 방식 OK
]:
    if has(c):
        team_diff_cols.append(c)

# (D) Herald/Horde는 보통 obj_* totals로 존재하므로 diff가 없으면 생성
# Herald
if has("obj_riftHerald_kills") and not has("riftHerald_kills_diff"):
    add_diff_from_total(df, "obj_riftHerald_kills", "riftHerald_kills_diff")
if has("riftHerald_kills_diff"):
    team_diff_cols.append("riftHerald_kills_diff")

# Horde (Void Grubs) = horde
if has("obj_horde_kills") and not has("horde_kills_diff"):
    add_diff_from_total(df, "obj_horde_kills", "horde_kills_diff")
if has("horde_kills_diff"):
    team_diff_cols.append("horde_kills_diff")

team_diff_cols = sorted(set(team_diff_cols))
print("team_diff_cols:", team_diff_cols)
print("team_diff count:", len(team_diff_cols))

team_diff_cols: ['assists_diff', 'atakhan_kills_diff', 'baron_kills_diff', 'deaths_diff', 'dragon_kills_diff', 'gold_earned_diff', 'gold_spent_diff', 'horde_kills_diff', 'inhibitor_takedowns_diff', 'kills_diff', 'riftHerald_kills_diff', 'turret_plates_taken_diff', 'turret_takedowns_diff', 'vision_score_diff', 'wards_killed_diff', 'wards_placed_diff']
team_diff count: 16


In [6]:
positions = ["TOP","JNG","MID","BOT","SUP"]

# 원천 컬럼명(마스터 DM에 존재하는 형태: {metric}_{POS})
pos_metric_sources = [
    "kills",
    "deaths",
    "assists",
    "gold_earned",
    "gold_spent",
    "total_damage_dealt_to_champions",
    "total_damage_taken",
    "damage_self_mitigated",
    "total_minions_killed",      # lane_cs
    "neutral_minions_killed",    # jungle_cs
    "vision_score",
    "wards_placed",
    "wards_killed",
    "ch_kill_participation",
    "ch_solo_kills",
    "ch_multikills",
    "ch_takedowns",
]

# 표현용 alias
metric_alias = {
    "total_minions_killed": "lane_cs",
    "neutral_minions_killed": "jungle_cs",
    "total_damage_dealt_to_champions": "dmg_to_champ",
    "total_damage_taken": "dmg_taken",
    "damage_self_mitigated": "dmg_mitigated",
}

pos_diff_cols = []

for pos in positions:
    for src in pos_metric_sources:
        raw_col = f"{src}_{pos}"
        if raw_col not in df.columns:
            continue

        alias = metric_alias.get(src, src)
        new_col = f"pos_{pos}__{alias}__diff"

        # diff = our_pos_metric - enemy_pos_metric (match_id 2row)
        s = df.groupby("match_id")[raw_col].transform("sum")
        df[new_col] = df[raw_col] - (s - df[raw_col])
        pos_diff_cols.append(new_col)

pos_diff_cols = sorted(set(pos_diff_cols))
print("pos_diff count:", len(pos_diff_cols))
print("pos_diff sample:", pos_diff_cols[:15])

pos_diff count: 85
pos_diff sample: ['pos_BOT__assists__diff', 'pos_BOT__ch_kill_participation__diff', 'pos_BOT__ch_multikills__diff', 'pos_BOT__ch_solo_kills__diff', 'pos_BOT__ch_takedowns__diff', 'pos_BOT__deaths__diff', 'pos_BOT__dmg_mitigated__diff', 'pos_BOT__dmg_taken__diff', 'pos_BOT__dmg_to_champ__diff', 'pos_BOT__gold_earned__diff', 'pos_BOT__gold_spent__diff', 'pos_BOT__jungle_cs__diff', 'pos_BOT__kills__diff', 'pos_BOT__lane_cs__diff', 'pos_BOT__vision_score__diff']


In [7]:
# share 컬럼이 있으면 그걸 우선 사용
gold_share_cols = [f"gold_earned_share_{p}" for p in positions]
dmg_share_cols  = [f"total_damage_dealt_to_champions_share_{p}" for p in positions]
kill_share_cols = [f"kills_share_{p}" for p in positions]

share_ok = all(c in df.columns for c in gold_share_cols + dmg_share_cols + kill_share_cols)

if not share_ok:
    # share가 없다면 totals로 share를 계산해 만든다 (대체 로직)
    # 필요한 totals 컬럼이 있는지 확인
    needed_totals = []
    for p in positions:
        needed_totals += [
            f"gold_earned_{p}",
            f"total_damage_dealt_to_champions_{p}",
            f"kills_{p}",
        ]
    missing = [c for c in needed_totals if c not in df.columns]
    if missing:
        raise ValueError(f"Structure features need share columns OR totals columns. Missing totals: {missing[:10]}")

    # gold shares
    gsum = df[[f"gold_earned_{p}" for p in positions]].sum(axis=1).replace(0, np.nan)
    for p in positions:
        df[f"gold_earned_share_{p}"] = (df[f"gold_earned_{p}"] / gsum).fillna(0)

    # dmg shares
    dsum = df[[f"total_damage_dealt_to_champions_{p}" for p in positions]].sum(axis=1).replace(0, np.nan)
    for p in positions:
        df[f"total_damage_dealt_to_champions_share_{p}"] = (df[f"total_damage_dealt_to_champions_{p}"] / dsum).fillna(0)

    # kill shares
    ksum = df[[f"kills_{p}" for p in positions]].sum(axis=1).replace(0, np.nan)
    for p in positions:
        df[f"kills_share_{p}"] = (df[f"kills_{p}"] / ksum).fillna(0)

# 이제 share_cols 확보
gold_share_cols = [f"gold_earned_share_{p}" for p in positions]
dmg_share_cols  = [f"total_damage_dealt_to_champions_share_{p}" for p in positions]
kill_share_cols = [f"kills_share_{p}" for p in positions]

# 구조지표 5개 생성
df["struct__carry_dependency_gold"]   = df[gold_share_cols].max(axis=1)
df["struct__carry_dependency_damage"] = df[dmg_share_cols].max(axis=1)

df["struct__gold_concentration_index"]   = (df[gold_share_cols] ** 2).sum(axis=1)
df["struct__damage_concentration_index"] = (df[dmg_share_cols] ** 2).sum(axis=1)
df["struct__kill_concentration_index"]   = (df[kill_share_cols] ** 2).sum(axis=1)

struct_cols = [
    "struct__carry_dependency_gold",
    "struct__carry_dependency_damage",
    "struct__gold_concentration_index",
    "struct__damage_concentration_index",
    "struct__kill_concentration_index",
]
print("struct cols:", struct_cols)

struct cols: ['struct__carry_dependency_gold', 'struct__carry_dependency_damage', 'struct__gold_concentration_index', 'struct__damage_concentration_index', 'struct__kill_concentration_index']


In [8]:
feature_cols = context_cols + team_diff_cols + pos_diff_cols + struct_cols
feature_cols = [c for c in feature_cols if c in df.columns]  # 안전장치

df_feat = df[feature_cols].copy()
print("dm_match_team_feature shape:", df_feat.shape)
print("num columns:", len(df_feat.columns))

OUT_PATH = r"../../data/Riot_API/feature/datamart/dm_match_team_feature.csv"
df_feat.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH)

dm_match_team_feature shape: (108412, 114)
num columns: 114
saved: ../../data/Riot_API/feature/datamart/dm_match_team_feature.csv


In [9]:
# 새로 만든 diff들: pos_*__*__diff + (riftHerald/horde diff 생성했으면 그것들)
created_diff_cols = [c for c in df.columns if c.startswith("pos_") and c.endswith("__diff")]
for c in ["riftHerald_kills_diff", "horde_kills_diff"]:
    if c in df.columns:
        created_diff_cols.append(c)

# match_id별로 team 100/200 합이 0이어야 함
bad = []
for c in created_diff_cols:
    s = df.groupby("match_id")[c].sum()
    # 완전 0이 아닐 때(부동소수 오차 고려)
    if (s.abs() > 1e-6).any():
        bad.append(c)

print("created diff cols checked:", len(created_diff_cols))
print("bad diff cols (should be empty):", bad[:20])

created diff cols checked: 87
bad diff cols (should be empty): []


In [10]:
def feature_group(name: str) -> str:
    if name.startswith("pos_TOP__"): return "TOP"
    if name.startswith("pos_JNG__"): return "JNG"
    if name.startswith("pos_MID__"): return "MID"
    if name.startswith("pos_BOT__"): return "BOT"
    if name.startswith("pos_SUP__"): return "SUP"
    if name.startswith("struct__"):  return "STRUCT"
    if name.endswith("_diff"):       return "TEAM_DIFF"
    return "META"

catalog = pd.DataFrame({
    "feature_name": feature_cols,
    "group": [feature_group(c) for c in feature_cols],
    "ml_allowed": [0 if c in context_cols else 1 for c in feature_cols],
})

OUT_CAT = r"../../data/Riot_API/feature/datamart/dm_feature_catalog.csv"
catalog.to_csv(OUT_CAT, index=False)

print("saved:", OUT_CAT)
print(catalog["group"].value_counts())

saved: ../../data/Riot_API/feature/datamart/dm_feature_catalog.csv
group
JNG          17
BOT          17
SUP          17
MID          17
TOP          17
TEAM_DIFF    16
META          8
STRUCT        5
Name: count, dtype: int64
